# Temas 2 - pandas

## Bloque 1. que es un DataFrame

Es una tabla: filas y columnas con nombre, como una hoja de Excel o una tabla de SQL.

Cada Columna es una Serie -> una lista de valores con nombre y todos del mismo tipo. 

                  ┌────────── DataFrame (la tabla entera) ──────────┐
         índice   station      date            value    quality
           0      Stockholm    1730865600000   -1.4     G
           1      Stockholm    1730869200000   -2.0     G
           2      Malmö        1730865600000    2.1     Y
                               └── Series ──┘  (una columna)
        

Antes trabajamos en el Tema 1 trabajaste con listas de diccionarios, y cada operacion necesitaba un for:
- filtrar por calidad, convertir valores, calcular minimos.

Pandas trabaja con la tabla completa de una vez. Lo que llevo 15 lineas en Repaso 5, en pandas son 3 o 4. 

Es la herramienta estandar para analizar datos en Python, y las API de los DataFrames de Spark se le parece mucho. 

In [ ]:
import pandas as pd   # convención universal: siempre "pd"

mediciones = [
    {"station": "Stockholm", "date": 1730865600000, "value": "-1.4", "quality": "G"},
    {"station": "Stockholm", "date": 1730869200000, "value": "-2.0", "quality": "G"},
    {"station": "Malmö",     "date": 1730865600000, "value": "2.1",  "quality": "Y"},
]

df = pd.DataFrame(mediciones)   # lista de dicts → tabla: cada dict es una fila, cada clave una columna
df                              # en un notebook, la última línea de una celda se muestra sola, como tabla

In [ ]:
print(type(df))          # <class 'pandas.core.frame.DataFrame'>
print(type(df["value"])) # <class 'pandas.core.series.Series'> → una columna
print(df.shape)          # (3, 4) → (filas, columnas)
print(df.columns)        # los nombres de las columnas
print(df.dtypes)         # el tipo de cada columna

### Practica guiada - Paso 1

In [ ]:
respuesta_malmo = {
    "station": {"key": "52350", "name": "Malmö"},
    "value": [
        {"date": 1730887200000, "value": "2.1", "quality": "G"},
        {"date": 1730890800000, "value": "2.6", "quality": "Y"},
        {"date": 1730894400000, "value": "3.0", "quality": "G"},
    ],
}

In [ ]:
values_malmo = respuesta_malmo['value']
df_malmo = pd.DataFrame(values_malmo)
print(df_malmo.shape)
print(df_malmo.columns)        # los nombres de las columnas
print(df_malmo.dtypes)
df_malmo

### Practica guiada - Paso: la otra forma de crear un DF

In [ ]:
datos = {
    "ciudad": ["Visby", "Ystad", "Kiruna", "Lund"],
    "temp": [5.1, 6.0, -12.4, 6.3],
}
df_ciudades = pd.DataFrame(datos)
print(df_ciudades)
print(df_ciudades.shape)
print(df_ciudades.dtypes)
df_ciudades["pais"] = "SE"
print(df_ciudades.shape)
df_ciudades

In [ ]:
lecturas = [
    {"ciudad": "Visby", "temp": 5.1},
    {"ciudad": "Ystad"},
    {"ciudad": "Kiruna", "temp": -12.4, "viento": 7.2},
]
df_lecturas = pd.DataFrame(lecturas)
df_lecturas

## Bloque 2: leer y exportar datos

**pd.read_csv()** -> lee un archivo CSV y lo convierte en un DataFrame.

El primer paso con cualquier dato nuevo es explorarlo:

- cuantas filas tiene, que tipos tienen las columnas, si hay nullos y si hay valores raros. 

Saltarse esto es el origen de la mayoria de los bugs en pipelines. En Databricks haras lo mismo con **spark.read.csv** y **display()**

Un problema sueco -> los CSV suecos u muchos europeos usan ; como separador y , como decimal. 

Pandas por defecto espera lo contrario: , como reparador y . como decimal.

In [ ]:
df = pd.read_csv("datos/observaciones.csv", sep=";", decimal=",")
df.head()
df.shape     # las 5 primeras filas (df.head(10) para 10)

**Parametro** sep=";" -> que caracter separa las columnas
**Parametro** decimal="." -> que caracter separa los decimales

Las funciones para explorar:

**df.head() / df.tail()**	Las primeras / últimas filas	Ver qué pinta tienen los datos

**df.shape**	(filas, columnas)	Saber el tamaño

**df.dtypes**	El tipo de cada columna	Comprobar que los números son números

**df.info()**	Tipos + cuántos valores no nulos tiene cada columna	Detectar nulos de un vistazo

**df.describe()**	Estadísticas de las columnas numéricas: count, mean, std, min, cuartiles, max	Detectar valores raros (un 99 °C, por ejemplo)

### Practica guiada - Paso 3: el error tipico primero

In [ ]:
df_mal = pd.read_csv("datos/observaciones.csv")
df_mal.head()

In [ ]:
df_mal.shape

### Practica guiada - Paso 4: leer bien y explorar

1. Lee el archivo correctamente, con sep y decimal, en una variable df.
2. Antes de ejecutar las siguientes líneas, predice:
    - ¿Qué shape tendrá ahora?
    - ¿Qué dtype tendrá temperatur: object o float64? ¿Por qué?
    - En df.info(), ¿cuántos valores no nulos tendrá temperatur? Mira bien el CSV.
3. Ejecuta df.shape, df.info() y df.describe(), cada uno en su celda.

In [ ]:
df = pd.read_csv("datos/observaciones.csv", sep=";", decimal=",")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

### Bloque 3: seleccionar y filtrar

Quedarte solo con algunas columnas (como el SELECT de SQL) o con algunas filas que cumplen una condicion (como el WHERE).

Casi todo el analisis empieza asi:
- "Solo las mediciones de calidad G", "solo la temperatura y la hora"


In [ ]:
df["temperatur"]                  # UNA columna → Series

In [ ]:
df[["station", "temperatur"]]     # VARIAS columnas → DataFrame (fíjate en los DOBLES corchetes)

**Filtrat filas:** la mascara booleana

Filtrar en pandas se hace en dos pasos:

In [ ]:
# 1. Una Series de TRUE/FALSE: una por fila

mascara = df['kvalitet'] == "G"
mascara

In [ ]:
# 2. Solo las filas donde la mascara es TRUE
df[mascara]

In [ ]:
# lo mismo , en una sola linea (la forma mas habitual)
df[df['kvalitet'] == "G"]

**Combinar condiciones**

In [ ]:
df[(df['station'] == 'Kiruna') & (df['kvalitet'] == 'G')]

In [ ]:
df[(df['temperatur'] < -10) | (df['temperatur'] > 2)] # -> or

In [ ]:
df[~(df['kvalitet'] == 'G')] #NO (lo conrario)

In [ ]:
df[df['station'].isin(['Malmö', 'Kiruna'])] # esta en esta lista

**loc**: filas y columnas a la vez

In [ ]:
df.loc[df["station"] == "Kiruna", ["tid", "temperatur"]]
#      └──── qué filas ────┘   └──── qué columnas ────┘
# que filas tienen de nombre estacion Kiruna
# las columnas tid y temperatur de las filas que hemos seleccionado antes

### Practica guiada - Paso 5

Usando tu df:

1. Predice sin ejecutar cuántas filas devuelve cada línea:

In [ ]:
# df[df["kvalitet"] == "G"]                                    # a)
# df[df["station"] == "Kiruna"]                                # b)
# df[(df["station"] == "Kiruna") & (df["temperatur"] < -13)]   # c)
# df[df["temperatur"] > 0]                                     # d)

a -> todas las filas que tengas kvalitet G -> 9 filas
b -> todas las filas que tengas de vaor Kiruna en la columna station -> 4
c -> todas las filas que tengan de valor Kiruna en la columna station, y tambien solo las filas donde el valor de temperatura se menos de 13 -> 2 filas
d -> todas las filas que tengas de valor en temperatur mas de 0 -> 3 filas

In [ ]:
df.loc[(df['station'] == 'Malmö') & (df["kvalitet"] == 'G'), ['tid' , 'temperatur']]

## Bloque 4a: nulos y duplicados

Es lo que se hace de pasar a bronze a silver en el MODULO3:tratarnulos,quitar duplicados , estandadrizar tipos.

In [ ]:
df.isna() # tabla de TRUE/FALSE: True donde hay NaN


In [ ]:
df.isna().sum() # cuantos NaN hay en cada columna

In [ ]:
df[df['temperatur'].isna()] # las filas con temperatura nula

In [ ]:
df[df['temperatur'].notna()]

**Hay dos formas de tratar nulos:**

df.dropna(subset=['temperatur'])

- Elimina las filas con NaN en esa columna
  
- Se usa cuando el dato es imprescindible y no tiene sentido invertarlo

df['temperatur'].fillna(valor)

- Rellena los NaN con un valor

- Hay un valor razonable de reemplazo(la media, la mediana, 0...)

**subset** dice en que columnas mirar. Sin **subset, dropna()** elimina cualquier fila que tenga algun NaN, en cualquier columna, y puedes perder filas sin darte cuenta. 


**Duplicados**


In [ ]:
df.duplicated() # True en las filas que repiten una anterios

In [ ]:
df.duplicated().sum() # cuants hay

In [ ]:
df.drop_duplicates() # sin las filas repetidas (identicas en TODO)

In [ ]:
df.drop_duplicates(subset=['station', 'datum', 'tid'])
#repetidas solo en estas columnas

**subset** en duplicados es clave. Si una estacion envia dos mediciones para la misma hora, aunque sean distintas, es un duplicado: una estacion no mide dos veces a la misma hora. Por eso se define que columnas identifican una medicion unica que en nuestro caso son : estacion + fecha + hora

**La trampa mas importante de pandas**

Estos metodos no modifican el DF original. Devuelve uno nuevo:

In [ ]:
df.dropna(subset=['temperatur'])
# crea una tabla nueva... y despues se pierde si no se guarda


In [ ]:
print(df.shape)

In [ ]:
df_limpio = df.dropna(subset=['temperatur'])

### Practica guiada - Paso 6

1. Predice sin ejecutar
   - ¿Qué devuelve df.isna().sum() para cada una de las 5 columnas?
   - ¿Qué devuelve df.duplicated().sum()?

devolvera 1 ya que solo hay un NaN en todo el dataset
devolvera 1 por que hay un duplicado que es la file 8

2. Crea df_limpio a partir de df, encadenando estas operaciones, cada una guardada en df_limpio:
   - Quitar las filas con temperatur nula.
   - Quitar los duplicados según estación + fecha + hora.
   - Quedarte solo con calidad "G".

In [ ]:
df_limpio = df_limpio.drop_duplicates(subset=['station', 'datum', 'tid'])

In [ ]:
df_limpio = df_limpio[df_limpio['kvalitet'] == 'G']

In [ ]:
df_limpio

In [ ]:
df.shape

In [ ]:
df_limpio.shape

In [ ]:
df_limpio = df_limpio.reset_index(drop=True)
df_limpio

## Bloque 4b : tipos y fechas

**Convertir tipos**

In [ ]:
df['temperatur'].astype(float)
# convierte; Falla si hay un valor imposible de convertir

In [ ]:
pd.to_numeric(df_malmo["value"], errors="coerce")
# convierte; lo que no se pueda -> NaN, sin fallar

**Fechas: texto -> datetime**

Tus columnas datum y tid son texto. Para calcular con ellas (ordenar, extraer la hora, convertir zonas), hay que convertirlas

In [ ]:
df_limpio['fecha'] = pd.to_datetime(
    # une los dos textos "2024-11-06 05:00"
    df_limpio['datum'] + " " + df_limpio['tid'],
    # los mismos codigos que strptime
    format="%Y-%m-%d %H:%M",
)
df_limpio

**Fechas: epoch en ms** -> datetime UTC (lo de SMHI)

In [ ]:
pd.to_datetime(df_malmo['date'], unit='ms', utc=True)

**El accesor .dt**

Para usar funciones de fecha sobre una columna entera, se pone .dt delante

In [ ]:
df_limpio['fecha'].dt.hour # la hora de cada fila

In [ ]:
df_limpio["fecha"].dt.strftime("%H:%M")   # texto, como en el Tema 1

In [ ]:
df_limpio["fecha"].dt.day_name() #'Wednesday'...

### Practica guiada - Paso 7

**Parte A** , con df_limpio:
1. Crea la columna fecha uniendo datum y tid, convertida a datetime.
2. Las horas del CSV son hora sueca. Crea fecha_utc con esas fechas convertidas a UTC. Piensa: ¿qué método va primero, tz_localize o tz_convert? ¿Hacen falta los dos?
3. Predice qué hora UTC tendrá la fila de Stockholm 05:00 (recuerda: en noviembre, Suecia está en UTC+1).

In [ ]:
df_limpio

In [ ]:
fechas_utc = df_limpio['fecha']
fechas_utc = fechas_utc.dt.tz_localize('Europe/Stockholm')

In [ ]:
df_limpio["fecha_utc"] = df_limpio["fecha"].dt.tz_localize("Europe/Stockholm").dt.tz_convert("UTC")

In [ ]:
df_limpio[["station", "fecha", "fecha_utc"]]

**Parte B** epoch de SMHI → hora sueca, con df_malmo

Trabajas con df_malmo del Paso 1, que tiene la columna date con los epochs en milisegundos.

1. Predice primero, sin ejecutar, las tres horas en formato HH:MM, hora sueca. Pista: son los mismos epochs del Paso 7 del Tema 1, que en UTC eran 10:00, 11:00 y 12:00.
2. Crea en df_malmo una columna nueva llamada hora_suecia, encadenando tres eslabones, sin for:
    - Eslabón 1: epoch en ms → datetime en UTC, con pd.to_datetime(...) y los parámetros de la unidad y de UTC.
    - Eslabón 2: UTC → hora sueca (aquí no hace falta tz_localize, porque ya tiene la etiqueta UTC).
    - Eslabón 3: fecha → texto HH:MM, con el método de .dt que da formato.
3. Muestra df_malmo con su nueva columna y compara con tu predicción.

In [ ]:
df_malmo

In [ ]:
df_malmo['fecha'] = pd.to_datetime(df_malmo["date"], unit="ms", utc=True).dt.tz_convert('Europe/Stockholm').dt.strftime("%H:%M")

In [ ]:
df_malmo

In [ ]:
# df_malmo["fecha_utc"] = pd.to_datetime(df_malmo["date"], unit="ms", utc=True)

# df_malmo["hora_suecia"] = (
#     df_malmo["fecha_utc"]
#     .dt.tz_convert("Europe/Stockholm")
#     .dt.strftime("%H:%M")
# )

## Bloque 5: transformar

Crear columnas nuevas a partir de las que ya tienes, cambiar nombres, quitar columnas y ordernar la tabla. 

Casi nunca usas los datos tal como llegan. Necesitas columnas derivadas:
- estas bajo cero?, temperatura en Hahrenheir para un cliente EEUU, la hora del dia para agrupar

En el Modulo 3 haremos lo mismo con PySpark con withColumn

**Columnas calculadas (vectorizacion)**

In [ ]:
df_limpio['temp_f'] = df_limpio['temperatur'] * 9/ 5 + 32
# Operacion aritmetica con toda la columna
df_limpio["bajo_cero"] = df_limpio["temperatur"] < 0
# una comparacion -> columna de True/False
df_limpio["hora"] = df_limpio["fecha"].dt.hour
#extraer una parte de una fecha

In [ ]:
df_limpio

**Columnas con una condicion: np.where**

Cuando el valor depende de una condicion, "si pasa esto. pon A sino pon B"

In [ ]:
import numpy as np

In [ ]:
df_limpio["estado"] = np.where(df_limpio["temperatur"] < 0, "helada", "normal")
#                              └──── condición ────────┘  └ si True ┘ └ si False ┘

In [ ]:
df_limpio['estado']

**El accesor .str**

Igual que .dt para las fechas, .str da acceso a los metodos de texto sobre una columna entera:

In [ ]:
df_limpio["station"].str.upper()
# 'STOCKHOLM', 'MALMÖ'…

In [ ]:
df_limpio["station"].str.len()
# longitud de cada nombre

In [ ]:
df_limpio["station"].str.startswith("K")
# True / False por fila

**Renombrar y quitar columnas**

In [ ]:
df_limpio = df_limpio.rename(columns={
    "station": "estacion", "temperatur": "temp"})
 # {viejo: nuevo}

In [ ]:
df_limpio = df_limpio.drop(columns=["datum", "tid"])
# quitar columnas

In [ ]:
df_limpio

**Ordenar**

In [ ]:
df_ordenado = df_limpio.sort_values("temp")
 # de menor a mayor

In [ ]:
df_limpio.sort_values("temp", ascending=False)
# de mayor a menor

In [ ]:
df_limpio.sort_values(["estacion", "fecha"])
# por estación y, dentro de cada una, por fecha

### Practica guiada - Paso 8 

Con df_limpio (el que tiene fecha y fecha_utc):

1. Crea la columna hora con la hora en hora sueca (un número), sacándola de fecha.
2. Crea la columna estado: "helada" si la temperatura es menor o igual a 0, y "ok" si no.
3. Renombra station → estacion y temperatur → temp.
4. Quita las columnas datum, tid y kvalitet, que ya no necesitas.
5. Ordena de la temperatura más baja a la más alta, y guarda el resultado en df_ordenado.

In [ ]:
df_ordenado

In [ ]:
df_ordenado = df_ordenado.drop(columns=['estado'])


In [ ]:
df_ordenado["estado"] = np.where(df_ordenado["temp"] <= 0, "helada", "ok")


In [ ]:
df_ordenado = df_ordenado.drop(columns=['temp_f', 'bajo_cero'])


In [ ]:
df_ordenado

## Bloque 6: agrupar

Dividir la tabla en grupos (por ejemplo, por estacion), calcular algo en cada grupo(min, avg, conteo) y juntas los resultado en una tabla nueva.

Se llama **split -> apply -> combine**

         split (por estación)         apply (min)        combine
Kiruna    -13.1, -12.7, -12.4    →      -13.1     ┐
Stockholm  -2.0,  -1.4           →       -2.0     ├→  tabla: estacion | min
Malmö       2.6,   3.0           →        2.6     ┘

Es el corazon de cualquier informe: 
- "temperatura media por ciudad"
- "ventas por mes"
- "clientes por region"

Es el GROUP BY de SQL, y en PySpark, groupBy().agg()


**Una agregacion**

In [ ]:
df_limpio.groupby('estacion')['temp'].mean()
#          └─ agrupa por ─┘   └ columna ┘ └ qué calcular ┘

In [ ]:
resumen = df_limpio.groupby("estacion").agg(
    minima=("temp", "min"),          # nombre_nuevo=(columna, función)
    maxima=("temp", "max"),
    media=("temp", "mean"),
    n=("temp", "count"),
)

In [ ]:
resumen

In [ ]:
resumen = resumen.reset_index()

In [ ]:
resumen

In [ ]:
df_limpio.groupby(["estacion", "estado"]).size().reset_index(name="n")

### Practica guiada - Paso 9

Antes de nada: recrea estado en df_limpio (con <= 0, "helada" y "ok"), y comprueba con df_limpio.groupby(["estacion", "estado"]).size() que Malmö sale como ok.

1. Predice sin ejecutar los valores de Kiruna: mínima, máxima, media (1 decimal) y número de mediciones.
2. Crea resumen, con una fila por estación y las columnas minima, maxima, media, n y horas_helada.
3. Redondea media a 1 decimal, aplica reset_index() y ordena de la estación más fría a la más cálida, por media.

In [ ]:
df_limpio = df_limpio.drop(columns=['estado'])


In [ ]:
df_limpio["estado"] = np.where(df_limpio["temp"] <= 0, "helada", "ok")


In [ ]:
df_limpio.groupby(["estacion", "estado"]).size()

In [ ]:
df_limpio

In [ ]:
resumen = df_limpio.groupby("estacion").agg(
    minima=('temp', 'min'),
    maxima=('temp', 'max'),
    media=('temp','mean'),
    n=('temp', 'count'),
    horas_helada=('bajo_cero','sum')
).round(1).sort_values('media').reset_index()

In [ ]:
resumen

## Bloque 7: unir tablas con merge

Combinar dos tablas usando una columna comun(la clave), como un JOIN de SQL.

Los datos casi nunca estan en un sola tabala. Las mediciones por un lado, la unformacion de las estaciones(region, altura) por otro lafo, y las poblacion de las SCB por otro. En el Modulo 3 haras joins en PySpark, que es tema de examn. 

**Ejemplo**

In [ ]:
estaciones = pd.DataFrame({
    "estacion": ["Stockholm", "Kiruna", "Umeå"],
    "region": ["Stockholm", "Norrbotten", "Västerbotten"],
    "altura_m": [44, 452, 14],
})

resumen.merge(estaciones, on="estacion", how="inner")
#             └ tabla derecha ┘  └ clave ┘  └ tipo de join ┘

In [ ]:
resumen

### Practica guiada - Paso 10: solo prediccion

Como te propuse, en este paso solo predices. No ejecutes nada hasta que me envíes tus respuestas.

Con resumen (3 filas: Kiruna, Stockholm, Malmö) y la tabla estaciones del ejemplo:

1. ¿Cuántas filas devuelve cada how: inner, left, right y outer?
2. En el left, ¿qué fila tendrá NaN, y en qué columnas?
3. En el right, ¿qué fila tendrá NaN, y en qué columnas?
4. En el left, ¿qué dtype tendrá altura_m: int64 o float64? ¿Por qué? Pista: recuerda qué tipo de dato es NaN, del Bloque 1.

inner - dos filas; left - 3 filas; rigth - 3 filas; outer 4 filas

left -> malmo tendria nan

rigth -> umeo tendria nan

left -> float64

In [ ]:
inner = resumen.merge(estaciones, on="estacion", how="inner")
inner

In [ ]:
left = resumen.merge(estaciones, on="estacion", how="left")
left

In [ ]:
right = resumen.merge(estaciones, on="estacion", how="right")
right

In [ ]:
outer = resumen.merge(estaciones, on="estacion", how="outer")
outer

In [ ]:
outer = resumen.merge(estaciones, on="estacion", how="outer", indicator=True)
outer

## Bloque 8: guardar resultados

Escribir en un archivo para guardar o compartirlo

In [ ]:
df_limpio.to_csv("datos/limpio.csv", index=False)       # CSV
df_limpio.to_parquet("datos/limpio.parquet", index=False)  # Parquet (necesita pyarrow)

pd.read_parquet("datos/limpio.parquet")                  # leerlo de vuelta

### Practica guiada - Paso 11

Vas a guardar df_limpio (con sus columnas fecha y fecha_utc) en CSV y en Parquet, y a leer los dos archivos de vuelta.

Antes de ejecutar nada, predice:

1. Después de leer el CSV de vuelta, ¿qué dtype tendrá fecha_utc?
2. Después de leer el Parquet de vuelta, ¿qué dtype tendrá fecha_utc?
3. ¿Qué archivo crees que pesará más en disco, con solo estas 7 filas? (Esta tiene trampa.)

Envíame solo las tres predicciones. Después lo ejecutamos.

1 -> object

2 -> datime

3 -> csv

In [ ]:
import os

# Leer de vuelta
df_desde_csv = pd.read_csv("datos/limpio.csv")
df_desde_parquet = pd.read_parquet("datos/limpio.parquet")

print("CSV:", df_desde_csv["fecha_utc"].dtype)
print("Parquet:", df_desde_parquet["fecha_utc"].dtype)

# Tamaño en disco (en bytes)
print("CSV:", os.path.getsize("datos/limpio.csv"), "bytes")
print("Parquet:", os.path.getsize("datos/limpio.parquet"), "bytes")